###Step 1 — Importing Web Scraping Libraries
Web scraping is the process of automatically extracting information from websites.

In [2]:
import requests       #Downloads webpage content using HTTP requests
import pandas as pd        #Organizes extracted records into DataFrames for analysis
from bs4 import BeautifulSoup    #Parses HTML and helps extract information from webpage elements

print("Libraries imported successfully!")

Libraries imported successfully!


###Step 2 — Fetching the SpaceX Wikipedia Webpage
Now we'll use Python to retrieve the HTML content of Wikipedia's Falcon 9 launch history page.

In [3]:
# Step 2: Fetch Wikipedia webpage

url = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"

response = requests.get(                #Sends an HTTP GET request to retrieve the webpage
    url,
    headers={"User-Agent": "SpaceXCapstoneLearning/1.0 (educational project)"},
    timeout=30
)

print("HTTP Status Code:", response.status_code)
print("HTML Content Length:", len(response.text))

response.raise_for_status()

HTTP Status Code: 200
HTML Content Length: 3181178


###Step 3 — Parsing HTML with BeautifulSoup
Now we'll convert the downloaded HTML into a structure that Python can navigate.

In [4]:
# Step 3: Parse Wikipedia HTML

soup = BeautifulSoup(response.text, "html.parser")

# Find all HTML tables
tables = soup.find_all("table")

print("Total HTML tables found:", len(tables))

# Inspect the first five tables
for i, table in enumerate(tables[:5]):
    print(f"\nTable {i + 1}")

    headers = [
        th.get_text(" ", strip=True)
        for th in table.find_all("th")[:8]
    ]

    print("Headers:", headers)

Total HTML tables found: 19

Table 1
Headers: ['List of Falcon 9 and Falcon Heavy launches']

Table 2
Headers: ['Flight No.', 'Date and time ( UTC )', 'Version, booster [ i ]', 'Launch site', 'Payload [ j ]', 'Payload mass', 'Orbit', 'Customer']

Table 3
Headers: ['Flight No.', 'Date and time ( UTC )', 'Version, booster [ i ]', 'Launch site', 'Payload [ j ]', 'Payload mass', 'Orbit', 'Customer']

Table 4
Headers: ['Date and time ( UTC )', 'Version, booster [ i ]', 'Launch site', 'Payload [ j ]', 'Orbit', 'Customer']

Table 5
Headers: ['Date and time ( UTC )', 'Version, booster [ i ]', 'Launch site', 'Payload [ j ]', 'Orbit', 'Customer']


###Step 4 — Inspecting Launch Table Rows
Before extracting data, let's inspect the structure of Table 2. Wikipedia launch tables can contain additional rows for mission descriptions, so we need to distinguish actual launch records from explanatory text.

In [5]:
# Step 4: Inspect the structure of the first launch table

launch_table = tables[1]

rows = launch_table.find_all("tr")

print("Total rows in Table 2:", len(rows))

# Inspect the first six rows
for i, row in enumerate(rows[:6]):

    cells = row.find_all(["th", "td"], recursive=False)

    values = [
        cell.get_text(" ", strip=True)[:100]
        for cell in cells
    ]

    print(f"\nRow {i + 1}:")
    print("Number of cells:", len(cells))
    print("Values:", values)

Total rows in Table 2: 332

Row 1:
Number of cells: 10
Values: ['Flight No.', 'Date and time ( UTC )', 'Version, booster [ i ]', 'Launch site', 'Payload [ j ]', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']

Row 2:
Number of cells: 10
Values: ['418', 'January 4, 2025 01:27 [ 29 ]', 'F9 B5 B1073 ‑ 20', 'Cape Canaveral , SLC ‑ 40', 'Thuraya 4-NGS', '5,000 kg (11,000 lb)', 'GTO', 'Thuraya', 'Success', 'Success ( ASOG )']

Row 3:
Number of cells: 1
Values: ['Planned replacement for Thuraya 2 and 3. [ 30 ] [ 31 ]']

Row 4:
Number of cells: 10
Values: ['419', 'January 6, 2025 20:43 [ 32 ]', 'F9 B5 B1077 ‑ 17', 'Cape Canaveral , SLC ‑ 40', 'Starlink : Group 6 ‑ 71', '~ 17,500 kg (38,600 lb)', 'LEO', 'SpaceX', 'Success', 'Success ( JRTI )']

Row 5:
Number of cells: 1
Values: ['Launch of 24 Starlink v2 mini satellites to a 559 km (347 mi) orbit at an inclination of 43° to expa']

Row 6:
Number of cells: 10
Values: ['420', 'January 8, 2025 15:27 [ 33 ]', 'F9 B5 B1086 

###Step 5 — Extracting Launch Records Using BeautifulSoup
Now we'll extract the ten columns from Table 2 into a pandas DataFrame.

In [6]:
# Step 5: Extract launch records from Table 2

column_names = [
    "FlightNumber",
    "DateTime",
    "BoosterVersion",
    "LaunchSite",
    "Payload",
    "PayloadMass",
    "Orbit",
    "Customer",
    "LaunchOutcome",
    "BoosterLanding"
]

launch_records = []

for row in rows[1:]:

    cells = row.find_all(["th", "td"], recursive=False)

    # Only extract rows with exactly 10 cells
    if len(cells) != 10:
        continue

    values = [
        cell.get_text(" ", strip=True)
        for cell in cells
    ]

    # Keep rows with a numerical flight number
    if not values[0].isdigit():
        continue

    launch_records.append(values)

scraped_df = pd.DataFrame(
    launch_records,
    columns=column_names
)

print("Extracted launch records:", len(scraped_df))
print("Dataset shape:", scraped_df.shape)

display(scraped_df.head())

Extracted launch records: 165
Dataset shape: (165, 10)


,FlightNumber,DateTime,BoosterVersion,LaunchSite,Payload,PayloadMass,Orbit,Customer,LaunchOutcome,BoosterLanding
0,418,"January 4, 2025 01:27 [ 29 ]",F9 B5 B1073 ‑ 20,"Cape Canaveral , SLC ‑ 40",Thuraya 4-NGS,"5,000 kg (11,000 lb)",GTO,Thuraya,Success,Success ( ASOG )
1,419,"January 6, 2025 20:43 [ 32 ]",F9 B5 B1077 ‑ 17,"Cape Canaveral , SLC ‑ 40",Starlink : Group 6 ‑ 71,"~ 17,500 kg (38,600 lb)",LEO,SpaceX,Success,Success ( JRTI )
2,420,"January 8, 2025 15:27 [ 33 ]",F9 B5 B1086 ‑ 3,"Kennedy , LC ‑ 39A",Starlink : Group 12-11 (21 satellites),"~ 16,500 kg (36,400 lb)",LEO,SpaceX,Success,Success ( ASOG )
3,421,"January 10, 2025 03:53 [ 34 ]",F9 B5 B1071 ‑ 22,"Vandenberg , SLC ‑ 4E",NROL-153 (22 Starshield satellites) [ 35 ],Unknown,LEO,NRO,Success,Success ( OCISLY )
4,422,"January 10, 2025 19:11 [ 36 ]",F9 B5 B1067 ‑ 25,"Cape Canaveral , SLC ‑ 40",Starlink : Group 12-12 (21 satellites),"~ 16,500 kg (36,400 lb)",LEO,SpaceX,Success,Success ( JRTI )


###Step 6 — Validating Extracted Launch Records
Before cleaning, we must check whether our extraction missed any launch records.

In [7]:
# Step 6: Validate extracted launch records

print("Dataset shape:", scraped_df.shape)

# Convert flight numbers to integers
flight_numbers = scraped_df["FlightNumber"].astype(int)

print("\nMinimum flight number:", flight_numbers.min())
print("Maximum flight number:", flight_numbers.max())

# Check duplicate flight numbers
print("\nDuplicate flight numbers:",
      flight_numbers.duplicated().sum())

# Check missing flight numbers
expected_numbers = set(
    range(flight_numbers.min(), flight_numbers.max() + 1)
)

actual_numbers = set(flight_numbers)

missing_numbers = sorted(
    expected_numbers - actual_numbers
)

print("\nMissing flight numbers:", missing_numbers)
print("Total missing flight numbers:", len(missing_numbers))

# Check missing values
print("\nMissing values:")
print(scraped_df.isnull().sum())

# Inspect landing outcomes
print("\nBooster landing categories:")
print(scraped_df["BoosterLanding"].value_counts())

Dataset shape: (165, 10)

Minimum flight number: 418
Maximum flight number: 582

Duplicate flight numbers: 0

Missing flight numbers: []
Total missing flight numbers: 0

Missing values:
FlightNumber      0
DateTime          0
BoosterVersion    0
LaunchSite        0
Payload           0
PayloadMass       0
Orbit             0
Customer          0
LaunchOutcome     0
BoosterLanding    0
dtype: int64

Booster landing categories:
BoosterLanding
Success ( OCISLY )    56
Success ( ASOG )      46
Success ( JRTI )      41
Success ( LZ ‑ 4 )     8
Success ( LZ ‑ 1 )     6
Success ( LZ ‑ 2 )     5
No attempt             2
Failure ( JRTI )       1
Name: count, dtype: int64


###Step 7 — Cleaning the scraped dataset
Now we enter the data preprocessing stage.

Before converting dates, payload masses, or landing outcomes, we'll create a separate working DataFrame and inspect the values that require cleaning.

In [8]:
# Step 7: Create a working copy of scraped data

clean_df = scraped_df.copy()

# Convert flight numbers to integers
clean_df["FlightNumber"] = clean_df["FlightNumber"].astype(int)

# Inspect raw date formats
print("Sample launch dates:")
print(clean_df["DateTime"].head(10).to_string(index=False))

# Inspect payload mass values
print("\nSample payload masses:")
print(clean_df["PayloadMass"].head(10).to_string(index=False))

# Identify non-numeric payload mass entries
print("\nPayload mass values containing Unknown or N/A:")
unknown_mass = clean_df[
    clean_df["PayloadMass"].str.contains(
        r"Unknown|N/A",
        case=False,
        na=False
    )
]

print("Records found:", len(unknown_mass))

display(
    unknown_mass[
        ["FlightNumber", "PayloadMass"]
    ].head(10)
)

Sample launch dates:
 January 4, 2025 01:27 [ 29 ]
 January 6, 2025 20:43 [ 32 ]
 January 8, 2025 15:27 [ 33 ]
January 10, 2025 03:53 [ 34 ]
January 10, 2025 19:11 [ 36 ]
January 13, 2025 16:47 [ 37 ]
January 14, 2025 19:09 [ 38 ]
January 15, 2025 06:11 [ 39 ]
January 21, 2025 05:24 [ 46 ]
January 21, 2025 15:45 [ 48 ]

Sample payload masses:
              5,000 kg (11,000 lb)
           ~ 17,500 kg (38,600 lb)
           ~ 16,500 kg (36,400 lb)
                           Unknown
           ~ 16,500 kg (36,400 lb)
           ~ 16,500 kg (36,400 lb)
               2,000 kg (4,400 lb)
               2,517 kg (5,549 lb)
~ 15,300 kg (33,700 lb) (Starlink)
           ~ 15,500 kg (34,200 lb)

Payload mass values containing Unknown or N/A:
Records found: 15


,FlightNumber,PayloadMass
3,421,Unknown
23,441,Unknown
29,447,Unknown [ k ]
32,450,Unknown
33,451,Unknown
40,458,Unknown
43,461,Unknown
45,463,Unknown [ k ]
76,494,Unknown [ k ]
112,530,Unknown


###Step 8 — Cleaning and Converting Launch Dates

In [9]:
# Step 8: Clean launch dates

import re

# Remove Wikipedia citation references
clean_df["DateTime"] = clean_df["DateTime"].str.replace(
    r"\[\s*\d+\s*\]",
    "",
    regex=True
).str.strip()

# Convert strings into datetime values
clean_df["DateTime"] = pd.to_datetime(
    clean_df["DateTime"],
    errors="coerce",
    utc=True
)

print("Date column data type:")
print(clean_df["DateTime"].dtype)

print("\nMissing or invalid dates:")
print(clean_df["DateTime"].isnull().sum())

print("\nFirst five cleaned dates:")
display(clean_df[["FlightNumber", "DateTime"]].head())

print("\nDate range:")
print("Earliest:", clean_df["DateTime"].min())
print("Latest:", clean_df["DateTime"].max())

Date column data type:
datetime64[ns, UTC]

Missing or invalid dates:
0

First five cleaned dates:


,FlightNumber,DateTime
0,418,2025-01-04 01:27:00+00:00
1,419,2025-01-06 20:43:00+00:00
2,420,2025-01-08 15:27:00+00:00
3,421,2025-01-10 03:53:00+00:00
4,422,2025-01-10 19:11:00+00:00



Date range:
Earliest: 2025-01-04 01:27:00+00:00
Latest: 2025-12-17 15:27:00+00:00


###Step 9 — Cleaning Payload Mass
Now we'll convert payload mass values into numerical kilograms.

In [10]:
# Step 9: Clean payload mass values

import re
import numpy as np

def extract_payload_mass(value):
    text = str(value).strip()

    # Extract the first mass expressed in kilograms
    match = re.search(r"([\d,]+(?:\.\d+)?)\s*kg\b", text)

    if match:
        return float(match.group(1).replace(",", ""))

    # Preserve unknown or unparseable masses as missing
    return np.nan


clean_df["PayloadMassKg"] = clean_df["PayloadMass"].apply(
    extract_payload_mass
)

print("Missing numerical payload masses:")
print(clean_df["PayloadMassKg"].isnull().sum())

print("\nFirst ten cleaned payload masses:")
display(
    clean_df[
        ["FlightNumber", "PayloadMass", "PayloadMassKg"]
    ].head(10)
)

print("\nPayload mass statistics:")
display(clean_df["PayloadMassKg"].describe())

Missing numerical payload masses:
15

First ten cleaned payload masses:


,FlightNumber,PayloadMass,PayloadMassKg
0,418,"5,000 kg (11,000 lb)",5000.0
1,419,"~ 17,500 kg (38,600 lb)",17500.0
2,420,"~ 16,500 kg (36,400 lb)",16500.0
3,421,Unknown,NaN
4,422,"~ 16,500 kg (36,400 lb)",16500.0
5,423,"~ 16,500 kg (36,400 lb)",16500.0
6,424,"2,000 kg (4,400 lb)",2000.0
7,425,"2,517 kg (5,549 lb)",2517.0
8,426,"~ 15,300 kg (33,700 lb) (Starlink)",15300.0
9,427,"~ 15,500 kg (34,200 lb)",15500.0



Payload mass statistics:


,PayloadMassKg
count,150.000000
mean,14390.366667
std,4541.291150
min,758.000000
25%,15350.000000
50%,16100.000000
75%,16675.000000
max,17500.000000


###Step 10 — Cleaning Booster Landing Outcomes
Now we'll create a binary column called LandingSuccess.

In [11]:
# Step 10: Create binary booster landing target

clean_df["LandingSuccess"] = np.where(
    clean_df["BoosterLanding"].str.startswith("Success"),
    1,
    0
)

print("Landing success distribution:")
print(clean_df["LandingSuccess"].value_counts())

print("\nLanding success percentage:")
print(
    clean_df["LandingSuccess"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nLanding outcomes and assigned classes:")
display(
    clean_df.groupby(
        ["BoosterLanding", "LandingSuccess"]
    ).size().reset_index(name="Count")
)

print("\nUpdated dataset shape:", clean_df.shape)

Landing success distribution:
LandingSuccess
1    162
0      3
Name: count, dtype: int64

Landing success percentage:
LandingSuccess
1    98.18
0     1.82
Name: proportion, dtype: float64

Landing outcomes and assigned classes:


,BoosterLanding,LandingSuccess,Count
0,Failure ( JRTI ),0,1
1,No attempt,0,2
2,Success ( ASOG ),1,46
3,Success ( JRTI ),1,41
4,Success ( LZ ‑ 1 ),1,6
5,Success ( LZ ‑ 2 ),1,5
6,Success ( LZ ‑ 4 ),1,8
7,Success ( OCISLY ),1,56



Updated dataset shape: (165, 12)


###Step 11 — Final Validation and Export
Now we'll save the cleaned web-scraped dataset so we can use it in later notebooks.

In [12]:
# Step 11: Validate and export cleaned dataset

from pathlib import Path

print("Final dataset shape:", clean_df.shape)

print("\nMissing values:")
print(clean_df.isnull().sum())

print("\nDuplicate flight numbers:")
print(clean_df["FlightNumber"].duplicated().sum())

print("\nLanding success distribution:")
print(clean_df["LandingSuccess"].value_counts())

# Create output directory
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)

# Save cleaned dataset
output_path = output_dir / "spacex_web_scraped_cleaned.csv"

clean_df.to_csv(output_path, index=False)

print("\nDataset saved successfully!")
print("File location:", output_path)

Final dataset shape: (165, 12)

Missing values:
FlightNumber       0
DateTime           0
BoosterVersion     0
LaunchSite         0
Payload            0
PayloadMass        0
Orbit              0
Customer           0
LaunchOutcome      0
BoosterLanding     0
PayloadMassKg     15
LandingSuccess     0
dtype: int64

Duplicate flight numbers:
0

Landing success distribution:
LandingSuccess
1    162
0      3
Name: count, dtype: int64

Dataset saved successfully!
File location: data/spacex_web_scraped_cleaned.csv


In [13]:
from google.colab import files

files.download("data/spacex_web_scraped_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>